In [1]:
import tensorflow as tf

for g in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(g, True)
print(tf.config.list_physical_devices('GPU'))

I0000 00:00:1791528719.343687   37377 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt
import numpy as np
from glob import glob

In [3]:
IMAGESHAPE = [224, 224, 3]
training_data = 'chest_xray/train'
testing_data = 'chest_xray/test'

In [4]:
vgg_model = VGG16(input_shape=IMAGESHAPE, weights='imagenet', include_top=False)

I0000 00:00:1791528729.535397   37377 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2145 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6


In [5]:
for each_layer in vgg_model.layers:
    each_layer.trainable = False

In [6]:
classes = glob('chest_xray/train/*')

In [7]:
flatten_layer = Flatten()(vgg_model.output)
prediction = Dense(len(classes), activation='softmax') (flatten_layer)

In [8]:
final_model = Model(inputs=vgg_model.input, outputs=prediction)
final_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │        50,178 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,764,866 (56.32 MB)

 Trainable params: 50,178 (196.01 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [9]:
final_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [10]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(rescale = 1./255, shear_range = 0.2, zoom_range = 0.2, horizontal_flip = True)
testing_datagen = ImageDataGenerator(rescale = 1. / 255)

In [11]:
trainig_set = train_datagen.flow_from_directory('chest_xray/train', target_size= (224, 224), batch_size=8, class_mode='categorical')

Found 5216 images belonging to 2 classes.


In [12]:
test_set = testing_datagen.flow_from_directory('chest_xray/test', target_size= (224, 224), batch_size=8, class_mode='categorical')

Found 624 images belonging to 2 classes.


In [13]:
fitted_model = final_model.fit(trainig_set, validation_data=test_set, epochs=5)

Epoch 1/5


I0000 00:00:1791528733.391005   37377 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1791528734.612430   37508 service.cc:153] XLA service 0x75bdf80330b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1791528734.612485   37508 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 2050, Compute Capability 8.6 (Driver: 13.2.0; Runtime: 12.5.0; Toolkit: 12.5.0; DNN: 9.27.0)
I0000 00:00:1791528734.721471   37508 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1791528735.771268   37508 cuda_dnn.cc:461] Loaded cuDNN version 92700
I0000 00:00:1791528735.809664   37508 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1586__.68
W0000 00:00:1791528747.112719   37508 bfc_allocator.cc:502] Allocator (GPU_0_bfc) ran out of memory trying to allocate 2.17GiB (rounded to 2326790656)requested by 

  1/652 ━━━━━━━━━━━━━━━━━━━━ 9:16:06 51s/step - accuracy: 0.3750 - loss: 0.7693

I0000 00:00:1791528784.479893   37508 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


652/652 ━━━━━━━━━━━━━━━━━━━━ 160s 167ms/step - accuracy: 0.9227 - loss: 0.2108 - val_accuracy: 0.8157 - val_loss: 0.7343
Epoch 2/5
652/652 ━━━━━━━━━━━━━━━━━━━━ 93s 142ms/step - accuracy: 0.9456 - loss: 0.1887 - val_accuracy: 0.9119 - val_loss: 0.3844
Epoch 3/5
652/652 ━━━━━━━━━━━━━━━━━━━━ 103s 159ms/step - accuracy: 0.9588 - loss: 0.1489 - val_accuracy: 0.8958 - val_loss: 0.4732
Epoch 4/5
652/652 ━━━━━━━━━━━━━━━━━━━━ 102s 157ms/step - accuracy: 0.9628 - loss: 0.1400 - val_accuracy: 0.9327 - val_loss: 0.3444
Epoch 5/5
652/652 ━━━━━━━━━━━━━━━━━━━━ 101s 155ms/step - accuracy: 0.9599 - loss: 0.1381 - val_accuracy: 0.8862 - val_loss: 0.6767


In [14]:
final_model.save('our_model.h5')

In [15]:
model = load_model('our_model.h5')
img = image.load_img('chest_xray/val/PNEUMONIA/person1954_bacteria_4886.jpeg', target_size=(224,224))
imagee = image.img_to_array(img) / 255.0
imagee = np.expand_dims(imagee, axis=0)
prediction = model.predict(imagee)
if prediction[0][0] > prediction[0][1]:
    print('Person is safe.')
else:
    print('Person is affected with Pneumonia.')
print(f'Prediction: {prediction}')

W0000 00:00:1791529306.996037   37505 bfc_allocator.cc:502] Allocator (GPU_0_bfc) ran out of memory trying to allocate 2.14GiB (rounded to 2295595520)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
I0000 00:00:1791529307.001740   37505 bfc_allocator.cc:1049] BFCAllocator dump for GPU_0_bfc
I0000 00:00:1791529307.001746   37505 bfc_allocator.cc:1056] Bin (256): 	Total Chunks: 48, Chunks in use: 48. 12.0KiB allocated for chunks. 12.0KiB in use in bin. 1.3KiB client-requested in use in bin.
I0000 00:00:1791529307.001753   37505 bfc_allocator.cc:1056] Bin (512): 	Total Chunks: 5, Chunks in use: 4. 2.8KiB allocated for chunks. 2.0KiB in use in bin. 2.0KiB client-requested in use in bin.
I0000 00:00:1791529307.001758   37505 bfc_allocator.cc:1056] Bin (1024): 	Total Chunks: 7, Chunks in use: 7. 7.2KiB allocated for chun

1/1 ━━━━━━━━━━━━━━━━━━━━ 44s 44s/step
Person is affected with Pneumonia.
Prediction: [[2.7131096e-06 9.9999726e-01]]
